#### Notebook 01: Bronze Layer — Raw Ingestion

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

### 1. Define path constants

In [0]:
RAW_PATH    = "workspace.new_project.online_retail_ecommerce_table"
BRONZE_TABLE = "workspace.new_project.bronze_online_retail"

### 2. Read raw table — keep EVERYTHING as-is

In [0]:
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .table(RAW_PATH)

### Fix invalid column name BEFORE writing to Delta

In [0]:
df_raw = df_raw.withColumnRenamed("Customer ID", "Customer_ID")

#### 3. Add ingestion metadata columns

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_bronze = df_raw \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file", lit(RAW_PATH))

###4. Show schema and row count

In [0]:
df_bronze.printSchema()
print(f"Row count: {df_bronze.count():,}")

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer_ID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)

Row count: 1,067,371


### 5. Write to Delta format (Bronze layer)

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(BRONZE_TABLE)